# CH04 · Laboratoire NB01 — Relations, corrélation et cointégration

**Questions directrices :** deux rendements peuvent-ils être corrélés sans spread stable ? Pourquoi une p-value ADF ordinaire sur un résidu n'est-elle pas la bonne conclusion d'Engle–Granger ?

**Mode d'emploi :** exécuter dans l'ordre. Toutes les données sont **synthétiques et générées localement**, sans réseau. Uniquement formation pour les diagnostics retenus. Les résultats servent à comprendre, pas à investir.

**Prérequis :** numpy, pandas, matplotlib, statsmodels. Trois familles fictives C (cointégration stable), N (rendements liés mais absence de cointégration), R (rupture tardive).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

SEED = 20261109
rng = np.random.default_rng(SEED)
n = 700
idx = pd.bdate_range('2024-01-02', periods=n)
innovation = rng.normal(0, .007, n)
x = 4.5 + np.cumsum(innovation)
e = np.zeros(n)
for t in range(1, n):
    e[t] = .66 * e[t-1] + rng.normal(0, .012)
trend_other = np.cumsum(rng.normal(0, .006, n))
y_good = -.52 + 1.2 * x + e
y_other = -.52 + 1.2 * x + trend_other
y_break = -.52 + 1.2 * x + e + (np.arange(n) >= 470) * .14
# Open(t) = clôture(t-1) * exp(gap(t)), sans futur; le gap est synthétique.
opens = {}
for k, logprice in [('A',x),('B_C',y_good),('B_N',y_other),('B_R',y_break)]:
    close = np.exp(logprice)
    g = rng.normal(0, .002, n)
    opening = np.r_[close[0], close[:-1] * np.exp(g[1:])]
    opens[k] = opening

df = pd.DataFrame({
  'log_A':x,'log_B_C':y_good,'log_B_N':y_other,'log_B_R':y_break,
  'close_A':np.exp(x),'close_B_C':np.exp(y_good),'close_B_N':np.exp(y_other),'close_B_R':np.exp(y_break),
  'open_A':opens['A'],'open_B_C':opens['B_C'],'open_B_N':opens['B_N'],'open_B_R':opens['B_R']
}, index=idx)
FORM=400; VAL=550
print('Dataset synthétique :', df.shape, '| graine :', SEED)
print('Formation :',df.index[0].date(), '→',df.index[FORM-1].date())
print('Validation :',df.index[FORM].date(), '→',df.index[VAL-1].date())
print('Test :',df.index[VAL].date(), '→',df.index[-1].date())

## 1. Observer avant d’estimer — log-prix et rendements

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
for key in ['log_A','log_B_C','log_B_N']:
    axes[0].plot(df.index[:FORM],df[key].iloc[:FORM],label=key)
for key in ['log_B_C','log_B_N']:
    r=df[key].diff(); a=df['log_A'].diff()
    axes[1].plot(r.iloc[:FORM].rolling(60).corr(a.iloc[:FORM]),label=key+' ↔ log_A')
axes[0].legend();axes[0].set_title('Log-prix pendant la formation')
axes[1].legend();axes[1].set_title('Corrélation roulante des log-rendements (60 jours)')
plt.tight_layout();plt.show()

**Arrêt de compréhension.** Une corrélation roulante élevée permet-elle d’affirmer que les résidus d’une régression en niveaux sont stationnaires ? Justifier sans utiliser la figure seule.

## 2. Estimer OLS et appliquer les tests adaptés

In [ ]:
rows=[]; spreads={}; estimates={}
x_train=df['log_A'].iloc[:FORM]
for key in ['log_B_C','log_B_N','log_B_R']:
    y_train=df[key].iloc[:FORM]
    reg=sm.OLS(y_train,sm.add_constant(x_train)).fit()
    a,b=reg.params.iloc[0],reg.params.iloc[1]
    resid=y_train-a-b*x_train
    stat,pval,crit=coint(y_train,x_train,trend='c',autolag='aic')
    rows.append({'paire':key, 'corr_rend':round(y_train.diff().corr(x_train.diff()),3),
                 'beta':round(b,3),'R2':round(reg.rsquared,3),'p_Engle_Granger':pval,
                 'p_ADF_naif_sur_residu':adfuller(resid,regression='n',autolag='aic')[1]})
    spreads[key]=resid; estimates[key]=(a,b)
resultats=pd.DataFrame(rows).set_index('paire')
display(resultats.round(4))

**À interpréter :** pourquoi le p-value ADF de la dernière colonne n’est-elle pas le bon seuil pour la décision de cointégration ? Quelle paire doit être conservée comme candidat, et pourquoi la paire rompue n’est-elle pas automatiquement sûre ?

## 3. Comparer visuellement les résidus

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(12,8),sharex=True)
for a,(key,resid) in zip(axes,spreads.items()):
    a.plot(resid.index,resid.values,lw=1,label=key)
    a.axhline(resid.mean(),ls='--',color='black',alpha=.6)
    a.legend(loc='upper left')
plt.tight_layout();plt.show()

## 4. Persistance approximative du spread

In [ ]:
key='log_B_C'; s=spreads[key].values
model=sm.OLS(s[1:],sm.add_constant(s[:-1])).fit()
phi=model.params[1]
half=np.log(.5)/np.log(phi) if 0<phi<1 else np.nan
print('phi estimé :',round(phi,4),'| demi-vie AR(1) :',round(half,2),'séances')

**Attention :** la demi-vie prédit une décroissance de l’écart *attendu* dans un AR(1) fixé, pas la date de sortie effective d’un trade.

## 5. Rupture cachée : regarder au-delà de la formation

In [ ]:
# Uniquement à des fins d'analyse ex post ; ne pas réutiliser pour calibrer une stratégie.
a,b=estimates['log_B_R']
full_spread=df['log_B_R']-a-b*df['log_A']
plt.figure(figsize=(12,3));plt.plot(full_spread.index,full_spread,label='spread R');plt.axvline(df.index[470],color='darkred',ls='--',label='rupture générée');plt.legend();plt.show()

### Livrable NB01

Rédiger une note de 200–300 mots distinguant : corrélation, cointégration, stationnarité des résidus, rupture et limites d’un diagnostic sur formation.